# T08 — Đo thông lượng và quyết định bậc thang

T07 đã trả lời "một mẫu có vừa 16 GB không". Notebook này trả lời câu quyết định lịch chạy:
**với 30 giờ GPU mỗi tuần, trích đặc trưng cho cả bốn bộ có kịp không.**

Cách làm: chia mẫu theo bốn mức độ dài, đo 20 mẫu mỗi mức, rồi nhân trung vị từng mức với
phân bố độ dài thật của từng bộ. Kết quả ra số giờ GPU cho mỗi bộ, không phải một con số
ms/mẫu chung chung.

**Notebook settings trước khi chạy:**

- Accelerator: **GPU T4 x2**
- Internet: **On**
- Data: attach dataset `unicorn1209/vihallulens` — bắt buộc, cần cả bốn bộ để dựng phân bố
- Add-ons → Secrets: `HF_TOKEN` (không bắt buộc)

Ước tính khoảng **10–15 phút GPU**: nạp mô hình 2 phút, tokenize 71.520 mẫu 1–2 phút, đo 80
mẫu 5–8 phút.

Chạy hết từ trên xuống rồi copy output của **ô 3, ô 4 và ô 5** dán vào PR.

In [ ]:
# Ô 1 — lấy code. Chạy lại được nhiều lần: nếu thư mục đã có thì kéo bản mới về,
# vì `git clone` vào thư mục đã tồn tại sẽ hỏng và ta lặng lẽ chạy tiếp bằng code cũ.
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wsunicorn/vihallulens.git"
REPO_DIR = Path("/kaggle/working/vihallulens")


def run(*args, cwd=None):
    done = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(" ".join(args) + chr(10) + done.stdout + done.stderr)
    return done.stdout.strip()


if (REPO_DIR / ".git").is_dir():
    run("git", "fetch", "--quiet", "origin", cwd=REPO_DIR)
    run("git", "reset", "--quiet", "--hard", "origin/main", cwd=REPO_DIR)
    print("đã cập nhật repo có sẵn")
else:
    run("git", "clone", "--quiet", REPO_URL, str(REPO_DIR))
    print("đã clone mới")

%cd /kaggle/working/vihallulens
print("commit:", run("git", "log", "--oneline", "-1", cwd=REPO_DIR))

In [ ]:
# Ô 2 — cài đặt. Không cài lại torch: image Kaggle đã có bản dựng theo đúng CUDA của máy.
!pip install -q --no-deps -e .
!pip install -q -U bitsandbytes accelerate transformers pytest

In [ ]:
# Ô 3 — kiểm tra môi trường và toán dự báo, trên CPU. Ô này hỏng thì DỪNG, đừng đốt quota.
# Chạy bằng tiến trình riêng chứ không import trong kernel: `pip install -e .` ghi một file
# .pth mà Python chỉ đọc lúc khởi động, nên kernel đang chạy sẵn có thể không thấy gói.
import os

try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN: đã nạp từ Kaggle Secrets")
except Exception:
    print("HF_TOKEN: không có, vẫn chạy được vì Qwen2.5 là mô hình mở")

get_ipython().system("python scripts/probe_env.py")
get_ipython().system("python -m pytest tests/test_throughput.py -q")

In [ ]:
# Ô 4 — thử khô: đọc đủ bốn bộ, dựng phân bố độ dài, chọn mẫu để đo. Không nạp mô hình.
# Nếu ô này ra bảng đúng thì mọi thứ ngoài GPU đã chạy được. Khoảng 2 phút.
# Copy output dán vào PR.
!python scripts/measure_throughput.py --dry-run

In [ ]:
# Ô 5 — T08. Đo thật trên GPU rồi ghi vào results/feasibility.jsonl.
# Copy TOÀN BỘ output dán vào PR — phần KẾT LUẬN là căn cứ quyết định lịch chạy.
!python scripts/measure_throughput.py --per-tier 20

In [ ]:
# Ô 6 — chỉ chạy nếu ô 5 kết luận CẦN QUYẾT ĐỊNH.
# Nấc 1 của bảng sáu nấc lùi ở mục 5 CLAUDE.md: hạ ngân sách token còn 2.048.
# Đo ở T05: chỉ cắt thêm 1,09 % mẫu ISE-DSC01. Đây là số thật, không phải ước lượng.
!python scripts/measure_throughput.py --per-tier 20 --max-context-tokens 2048

In [ ]:
# Ô 7 — lấy file kết quả về máy. results/ không được ghi ngược lên GitHub từ notebook,
# nên tải file này xuống rồi commit từ máy cá nhân.
import shutil

shutil.copy("results/feasibility.jsonl", "/kaggle/working/feasibility.jsonl")
with open("results/feasibility.jsonl", encoding="utf-8") as handle:
    print(handle.read())

## Đọc kết quả thế nào

Ba con số quyết định, theo thứ tự quan trọng:

1. **Giờ GPU cho hai bộ bắt buộc** (ViHallu + ISE-DSC01). Ngưỡng đặt ở **15 giờ**, tức nửa
   quota tuần, vì nửa còn lại phải dành cho E09 tinh chỉnh bộ mã hóa và các ablation.
2. **Mũ k** trong `chi phí ≈ độ dài^k`. Gần 2 nghĩa là ma trận chú ý chi phối, hạ ngân sách
   token xuống một nửa sẽ rẻ đi gần bốn lần. Gần 1 nghĩa là phần còn lại của mạng chi phối,
   nấc lùi 1 gần như không giúp gì và phải tính hướng khác.
3. **VRAM đỉnh theo mức**. Phải còn dư so với 14 GB ở mức dài nhất, vì đây mới là lúc chạy
   liên tiếp nhiều mẫu chứ không phải hai mẫu như T07.

Nếu ô 5 kết luận **CẦN QUYẾT ĐỊNH**, đừng tự chọn hướng: ghi số vào **Nhật ký chặn** cuối
`TASKS.md` rồi hỏi, theo quy tắc 4 mục 6 của `CLAUDE.md`. Lùi mô hình đọc chính là đổi quyết
định đã chốt ở mục 3 `CLAUDE.md`.